# Chapman ECG Classification with Optuna

Notebook này thực hiện:

1. Đọc dữ liệu Chapman từ file CSV.
2. Chia dữ liệu thành train, validation và test.
3. Huấn luyện mô hình `MultiScaleCNN`.
4. Tối ưu siêu tham số bằng Optuna.
5. Huấn luyện lại trên toàn bộ tập train.
6. Đánh giá Accuracy và Micro-F1 trên tập test.
7. Lưu siêu tham số tốt nhất vào `results_optuna.csv`.

> Đặt notebook sao cho đường dẫn `../data/chapman/csv/raw.csv` và các file `Chapman.py`, `MultiScaleCNN.py` có thể được import đúng.

## 1. Cài đặt thư viện

Bỏ dấu `#` nếu môi trường chưa cài các thư viện cần thiết.

In [ ]:
# %pip install torch pandas numpy scikit-learn optuna

## 2. Import thư viện

In [1]:
import gc
import os
import random
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, random_split

from Chapman import ChapmanTestDataset, ChapmanTrainDataset
from MultiScaleCNN import MultiScaleCNN

C:\Users\nguye\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Thiết lập seed và thiết bị

In [2]:
def set_seed(seed_value: int = 42) -> None:
    random.seed(seed_value)
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    np.random.seed(seed_value)

    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4060


## 4. Đọc và chia dữ liệu

In [ ]:
DATA_PATH = Path("../data/chapman/csv/raw.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file dữ liệu: {DATA_PATH.resolve()}"
    )

data = pd.read_csv(DATA_PATH, header=None)

train_data_pd, test_data_pd = train_test_split(
    data,
    test_size=0.2,
    random_state=SEED,
)

data_train = ChapmanTrainDataset(train_data_pd)
data_test = ChapmanTestDataset(test_data_pd)

print("Total samples:", len(data))
print("Train + validation samples:", len(data_train))
print("Test samples:", len(data_test))

## 5. Chia tập train thành train và validation

In [ ]:
VAL_RATIO = 0.2

n_total = len(data_train)
n_val = int(n_total * VAL_RATIO)
n_train = n_total - n_val

split_generator = torch.Generator().manual_seed(SEED)

train_set, val_set = random_split(
    data_train,
    [n_train, n_val],
    generator=split_generator,
)

print("Training samples:", len(train_set))
print("Validation samples:", len(val_set))

## 6. Cấu hình huấn luyện

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

# Trên Windows/Jupyter, num_workers=0 thường ổn định hơn.
# Có thể tăng lên 2, 4 hoặc 8 nếu môi trường hỗ trợ.
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

## 7. Hàm tạo DataLoader

In [ ]:
def create_loader(dataset, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

## 8. Hàm huấn luyện một epoch

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY).long()

        optimizer.zero_grad(set_to_none=True)

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = running_loss / max(1, len(loader))
    accuracy = correct / max(1, total)

    return average_loss, accuracy

## 9. Hàm đánh giá

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_predictions = []
    all_labels = []

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY).long()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    average_loss = running_loss / max(1, len(loader))
    accuracy = correct / max(1, total)

    micro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="micro",
        zero_division=0,
    )

    return average_loss, accuracy, micro_f1

## 10. Hàm xây dựng mô hình

In [ ]:
def build_model(params: dict) -> MultiScaleCNN:
    return MultiScaleCNN(
        input_channels=1,
        kernel_size=params["kernel_size"],
        dropout_rate=params["dropout_rate"],
        n_heads=params["n_heads"],
        output_dim=params["output_dim"],
        attention_num_layers=params["attention_num_layers"],
    ).to(device)

## 11. Optuna objective

In [ ]:
def objective(trial: optuna.Trial) -> float:
    model = None

    try:
        # Dùng cùng seed cho các trial để việc so sánh công bằng hơn.
        set_seed(SEED)

        params = {
            "lr": trial.suggest_float("lr", 1e-5, 1e-1, log=True),
            "dropout_rate": trial.suggest_float(
                "dropout_rate", 0.1, 0.5
            ),
            "batch_size": trial.suggest_categorical(
                "batch_size", [16, 32]
            ),
            "weight_decay": trial.suggest_float(
                "weight_decay", 1e-6, 1e-3, log=True
            ),
            "kernel_size": trial.suggest_categorical(
                "kernel_size", [5, 6, 7, 8, 9]
            ),
            "n_heads": trial.suggest_categorical(
                "n_heads", [4, 8, 16, 32]
            ),
            "output_dim": trial.suggest_categorical(
                "output_dim", [128, 256, 512, 1024]
            ),
            "attention_num_layers": trial.suggest_categorical(
                "attention_num_layers", [1, 2, 3, 4, 5]
            ),
        }

        train_loader = create_loader(
            train_set,
            batch_size=params["batch_size"],
            shuffle=True,
        )

        val_loader = create_loader(
            val_set,
            batch_size=params["batch_size"],
            shuffle=False,
        )

        model = build_model(params)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=params["lr"],
            weight_decay=params["weight_decay"],
            betas=(0.9, 0.99),
        )

        max_epochs = 20
        best_val_micro_f1 = 0.0

        for epoch in range(max_epochs):
            train_loss, train_acc = train_one_epoch(
                model,
                train_loader,
                optimizer,
                loss_fn,
            )

            val_loss, val_acc, val_micro_f1 = evaluate(
                model,
                val_loader,
                loss_fn,
            )

            # Optuna tối ưu Micro-F1, đúng với nội dung in kết quả.
            trial.report(val_micro_f1, step=epoch)

            if trial.should_prune():
                raise optuna.TrialPruned()

            best_val_micro_f1 = max(
                best_val_micro_f1,
                val_micro_f1,
            )

            trial.set_user_attr(
                "last_epoch_metrics",
                {
                    "train_loss": float(train_loss),
                    "train_accuracy": float(train_acc),
                    "val_loss": float(val_loss),
                    "val_accuracy": float(val_acc),
                    "val_micro_f1": float(val_micro_f1),
                },
            )

        return best_val_micro_f1

    except torch.cuda.OutOfMemoryError:
        print(f"Trial {trial.number} bị CUDA Out Of Memory.")
        raise optuna.TrialPruned()

    finally:
        if model is not None:
            del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 12. Chạy Optuna study

In [ ]:
def run_study(n_trials: int = 25) -> optuna.Study:
    sampler = optuna.samplers.TPESampler(seed=SEED)

    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5,
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name="chapman_multiscale_cnn",
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=1,
        gc_after_trial=True,
    )

    print("Best validation Micro-F1:", study.best_value)
    print("Best parameters:")
    print(study.best_params)

    return study

Chạy cell dưới đây để bắt đầu tìm kiếm siêu tham số. Có thể giảm `N_TRIALS` khi muốn kiểm tra nhanh.

In [ ]:
N_TRIALS = 50

study = run_study(n_trials=N_TRIALS)
best_params = study.best_params

## 13. Xem kết quả các trial

In [ ]:
trials_df = study.trials_dataframe()
trials_df.head()

## 14. Huấn luyện mô hình cuối cùng và đánh giá test

In [ ]:
def train_final_and_test(
    best_params: dict,
    num_epochs: int = 60,
):
    print("Best parameters:")
    print(best_params)

    set_seed(SEED)

    batch_size = best_params["batch_size"]

    full_train_loader = create_loader(
        data_train,
        batch_size=batch_size,
        shuffle=True,
    )

    test_loader = create_loader(
        data_test,
        batch_size=batch_size,
        shuffle=False,
    )

    model = build_model(best_params)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        betas=(0.9, 0.99),
    )

    training_history = []

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model,
            full_train_loader,
            optimizer,
            loss_fn,
        )

        training_history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_accuracy": train_acc,
            }
        )

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc * 100:.2f}%"
        )

    test_loss, test_acc, test_micro_f1 = evaluate(
        model,
        test_loader,
        loss_fn,
    )

    print()
    print(f"Test Accuracy: {test_acc * 100:.2f}%")
    print(f"Test Micro-F1: {test_micro_f1 * 100:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")

    result_row = {
        **best_params,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_micro_f1": test_micro_f1,
        "num_epochs": num_epochs,
    }

    pd.DataFrame([result_row]).to_csv(
        "results_optuna.csv",
        index=False,
    )

    pd.DataFrame(training_history).to_csv(
        "training_history.csv",
        index=False,
    )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "best_params": best_params,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
            "test_micro_f1": test_micro_f1,
            "num_epochs": num_epochs,
        },
        "chapman_multiscale_cnn_optuna.pt",
    )

    return model, pd.DataFrame(training_history), result_row

In [ ]:
final_model, history_df, final_result = train_final_and_test(
    best_params,
    num_epochs=60,
)

## 15. Xem kết quả cuối cùng

In [ ]:
pd.DataFrame([final_result])

## 16. Vẽ lịch sử huấn luyện

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"])
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss by Epoch")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history_df["epoch"],
    history_df["train_accuracy"] * 100,
)
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy (%)")
plt.title("Training Accuracy by Epoch")
plt.grid(True)
plt.show()

## Những lỗi đã được sửa khi chuyển sang notebook

- Hàm `evaluate()` ban đầu nhận tham số `device`, nhưng lúc gọi lại thiếu tham số này.
- Hàm `evaluate()` trả về ba giá trị nhưng trong Optuna objective chỉ nhận hai giá trị.
- Objective ban đầu tối ưu validation accuracy, trong khi phần kết quả lại ghi Micro-F1.
- `del model` có thể gây lỗi nếu mô hình chưa được khởi tạo trước khi xảy ra exception.
- `num_workers=8` có thể làm Jupyter trên Windows bị treo hoặc tạo nhiều process; notebook đặt mặc định bằng `0`.
- Notebook bổ sung lưu checkpoint, lịch sử huấn luyện và kết quả test.